In [2]:
# ============================================================
# WEEK 8: CHECK AVAILABLE DATASET VARIABLES
# ============================================================

print("Available dataset and loader variables:\n")

for name, value in globals().copy().items():
    if any(word in name.lower() for word in [
        "dataset", "loader", "train", "val", "valid"
    ]):
        try:
            print(name, ":", type(value).__name__, "| Length:", len(value))
        except:
            print(name, ":", type(value).__name__)

Available dataset and loader variables:

__loader__ : NoneType


In [4]:

from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

ROOT = Path.cwd().parent
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"

print("Project:", ROOT)
print("Results folder:", RESULTS_DIR.exists())
print("Models folder:", MODELS_DIR.exists())


Project: c:\Users\HIMANSHU\Desktop\minor project 1\Minor-Project
Results folder: True
Models folder: True


In [6]:

face_metadata_df = pd.read_csv(
    RESULTS_DIR / "final_face_metadata.csv"
)

face_metadata_df = face_metadata_df[
    face_metadata_df["file_exists"].astype(str).str.lower() == "true"
].copy()

train_df = face_metadata_df[
    face_metadata_df["split"] == "train"
].reset_index(drop=True)

val_df = face_metadata_df[
    face_metadata_df["split"] == "val"
].reset_index(drop=True)

test_df = face_metadata_df[
    face_metadata_df["split"] == "test"
].reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Train: 39450
Validation: 9870
Test: 12321


In [8]:

class DeepfakeDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["face_path"])

        if not image_path.is_absolute():
            image_path = ROOT / image_path

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            float(row["label"]),
            dtype=torch.float32
        )

        return image, label


In [9]:

IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [10]:

train_dataset = DeepfakeDataset(train_df, train_transform)
val_dataset = DeepfakeDataset(val_df, test_transform)
test_dataset = DeepfakeDataset(test_df, test_transform)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


Train: 39450
Validation: 9870
Test: 12321


In [11]:

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Device:", device)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))


Device: cuda
Train batches: 2466
Validation batches: 617
Test batches: 771


In [12]:

checkpoint_path = MODELS_DIR / "week4_resnet50.pth"

print("Checkpoint exists:", checkpoint_path.exists())
print("Checkpoint path:", checkpoint_path)



Checkpoint exists: True
Checkpoint path: c:\Users\HIMANSHU\Desktop\minor project 1\Minor-Project\models\week4_resnet50.pth


In [13]:

import torch.nn as nn
from torchvision import models

resnet50 = models.resnet50(weights=None)

num_features = resnet50.fc.in_features
resnet50.fc = nn.Linear(num_features, 1)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    checkpoint = checkpoint["state_dict"]
elif isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    checkpoint = checkpoint["model_state_dict"]

# Handle DataParallel checkpoints
checkpoint = {
    k.replace("module.", ""): v
    for k, v in checkpoint.items()
}

resnet50.load_state_dict(checkpoint)
resnet50 = resnet50.to(device)
resnet50.eval()

print("ResNet50 checkpoint loaded successfully!")
print("Device:", device)


ResNet50 checkpoint loaded successfully!
Device: cuda


In [15]:

import numpy as np

resnet50.eval()

val_probabilities = []
val_labels = []

with torch.no_grad():
    for i, (images, labels) in enumerate(val_loader):

        images = images.to(device)

        outputs = resnet50(images)
        probabilities = torch.sigmoid(outputs).squeeze(1)

        val_probabilities.extend(
            probabilities.cpu().numpy()
        )

        val_labels.extend(labels.numpy())

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(val_loader)} batches")

val_probabilities = np.array(val_probabilities)
val_labels = np.array(val_labels).astype(int)

print("\nValidation predictions completed!")
print("Total predictions:", len(val_probabilities))
print("Probability range:",
      val_probabilities.min(),
      "to",
      val_probabilities.max())

print("Labels:", np.unique(val_labels, return_counts=True))


Processed 100/617 batches
Processed 200/617 batches
Processed 300/617 batches
Processed 400/617 batches
Processed 500/617 batches
Processed 600/617 batches

Validation predictions completed!
Total predictions: 9870
Probability range: 3.3589536e-08 to 1.0
Labels: (array([0, 1]), array([1380, 8490]))


In [16]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.01):

    predictions = (
        val_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(val_labels, predictions),
        "Precision": precision_score(
            val_labels, predictions, zero_division=0
        ),
        "Recall": recall_score(
            val_labels, predictions, zero_division=0
        ),
        "F1-Score": f1_score(
            val_labels, predictions, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

best_result = threshold_df.loc[
    threshold_df["F1-Score"].idxmax()
]

best_threshold = float(best_result["Threshold"])

print("BEST VALIDATION THRESHOLD")
print(best_result)

print("\nDEFAULT THRESHOLD (0.50)")
print(
    threshold_df[
        threshold_df["Threshold"] == 0.50
    ]
)


BEST VALIDATION THRESHOLD
Threshold    0.140000
Accuracy     0.917427
Precision    0.922121
Recall       0.987397
F1-Score     0.953643
Name: 4, dtype: float64

DEFAULT THRESHOLD (0.50)
    Threshold  Accuracy  Precision    Recall  F1-Score
40        0.5  0.811246   0.944348  0.829446  0.883176


In [17]:

import json

threshold_path = RESULTS_DIR / "week8_best_threshold.json"

with open(threshold_path, "w") as f:
    json.dump({
        "best_threshold": best_threshold,
        "selection_metric": "F1-Score",
        "validation_f1": float(best_result["F1-Score"])
    }, f, indent=4)

print("Best threshold saved:", best_threshold)
print("File:", threshold_path)


Best threshold saved: 0.14
File: c:\Users\HIMANSHU\Desktop\minor project 1\Minor-Project\results\week8_best_threshold.json


In [18]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

resnet50.eval()

test_probabilities = []
test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = resnet50(images)
        probabilities = torch.sigmoid(outputs).squeeze(1)

        test_probabilities.extend(probabilities.cpu().numpy())
        test_labels.extend(labels.numpy())

test_probabilities = np.array(test_probabilities)
test_labels = np.array(test_labels).astype(int)

print("Test predictions:", len(test_probabilities))

for threshold in [0.50, best_threshold]:

    predictions = (
        test_probabilities >= threshold
    ).astype(int)

    print(f"\n{'='*45}")
    print("THRESHOLD:", threshold)
    print("Confusion Matrix:")
    print(confusion_matrix(test_labels, predictions))

    print("\nClassification Report:")
    print(classification_report(
        test_labels,
        predictions,
        target_names=["Real", "Fake"],
        zero_division=0
    ))

    print("ROC-AUC:",
          roc_auc_score(test_labels, test_probabilities))


Test predictions: 12321

THRESHOLD: 0.5
Confusion Matrix:
[[1140  571]
 [1799 8811]]

Classification Report:
              precision    recall  f1-score   support

        Real       0.39      0.67      0.49      1711
        Fake       0.94      0.83      0.88     10610

    accuracy                           0.81     12321
   macro avg       0.66      0.75      0.69     12321
weighted avg       0.86      0.81      0.83     12321

ROC-AUC: 0.8331040597211259

THRESHOLD: 0.14
Confusion Matrix:
[[  725   986]
 [  162 10448]]

Classification Report:
              precision    recall  f1-score   support

        Real       0.82      0.42      0.56      1711
        Fake       0.91      0.98      0.95     10610

    accuracy                           0.91     12321
   macro avg       0.87      0.70      0.75     12321
weighted avg       0.90      0.91      0.89     12321

ROC-AUC: 0.8331040597211259


In [19]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

final_results = []

for threshold in [0.50, best_threshold]:

    predictions = (test_probabilities >= threshold).astype(int)

    final_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(test_labels, predictions),
        "Precision_Fake": precision_score(test_labels, predictions, zero_division=0),
        "Recall_Fake": recall_score(test_labels, predictions, zero_division=0),
        "F1_Fake": f1_score(test_labels, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(test_labels, test_probabilities)
    })

final_results_df = pd.DataFrame(final_results)

output_path = RESULTS_DIR / "week8_threshold_comparison.csv"
final_results_df.to_csv(output_path, index=False)

print(final_results_df.to_string(index=False))
print("\nSaved:", output_path)



 Threshold  Accuracy  Precision_Fake  Recall_Fake  F1_Fake  ROC_AUC
      0.50  0.807645        0.939139     0.830443 0.881453 0.833104
      0.14  0.906826        0.913766     0.984731 0.947922 0.833104

Saved: c:\Users\HIMANSHU\Desktop\minor project 1\Minor-Project\results\week8_threshold_comparison.csv
